In [26]:
import torch
import torch.nn as nn

import torch.nn.functional as F
from transformers import AutoTokenizer , AutoModelForCausalLM
import math

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name ="gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [27]:
def compute_entropy(probs):
    return -torch.sum(probs * torch.log(probs +1e-10)).item()

def repetition_ratio(tokens):
    unique = len(set(tokens))
    total =len(tokens)
    return 1- (unique/total)

def top_k_f(logits,k):
    values,indices =torch.topk(logits,k)
    min_value = values[:,-1].unsqueeze(1)
    filtered_logits = torch.where(
    logits < min_value,
    torch.tensor(float('-inf')).to(logits.device),
    logits
    )
    return filtered_logits

def top_p_f(logits,p):
    sorted_logits,sorted_indices =torch.sort(logits,descending=True)
    probs = F.softmax(sorted_logits,dim =-1)
    cumulative_probs = torch.cumsum(probs,dim=-1)
    #mask tokens where cumulative prob >p
    sorted_mask = cumulative_probs >p

    sorted_mask[:,1:] = sorted_mask[:,:-1].clone()
    sorted_mask[:,0] = False
    sorted_logits[sorted_mask] =float('-inf')

    original_logits = torch.zeros_like(logits)
    original_logits.scatter_(1,sorted_indices,sorted_logits)
    return original_logits

In [28]:
def generate_manual(
    prompt,max_new_tokens=50,
    temperature=1.0,top_k=None,top_p=None):
    input_ids = tokenizer(prompt,return_tensors="pt").to(device)
    generated = input_ids["input_ids"]
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs= model(input_ids=generated)
            logits=outputs.logits[:,-1,:]

        #1 temperature scaling
        logits = logits/temperature
        #2 apply top k
        if top_k is not None:
            logits =top_k_f(logits,top_k)
        #3 apply top p
        if top_p is not None:
            logits =top_p_f(logits,top_p)

        probs =F.softmax(logits,dim=-1)

        entropy =compute_entropy(probs)

        next_token = torch.multinomial(probs,num_samples=1)

        generated = torch.cat((generated,next_token),dim=1)

    text =tokenizer.decode(generated[0],skip_special_tokens=True)
    return text,entropy

        
        
            

In [29]:
prompt ="Artificial Intelligence is"
configs =[
    {"temperature":0.2},
    {"temperature":0.7},
    {"temperature":1.2},
    {"temperature":0.8,"top_k":10},
    {"temperature":0.8,"top_p":0.9},
]
for config in configs:
    text,entropy=generate_manual(prompt,**config)
    tokens =tokenizer.encode(text)

    print("="*50)
    print("Config:",config)
    print("Entropy:",entropy)
    print("Repetition:",repetition_ratio(tokens))
    print("Output:\n",text)
    

Config: {'temperature': 0.2}
Entropy: 0.0034971823915839195
Repetition: 0.2777777777777778
Output:
 Artificial Intelligence is a new field of research that is being pursued by the University of California, Berkeley. The goal of the project is to develop a new type of artificial intelligence that can be used to solve problems in real-world situations.

The goal of the
Config: {'temperature': 0.7}
Entropy: 2.8517885208129883
Repetition: 0.2407407407407407
Output:
 Artificial Intelligence is a technology that is both simple and elegant in a way that will push the boundaries of what we can learn from it.

The article is part of the Media Matters Project's 'Asterisk of AI' series.

It's a
Config: {'temperature': 1.2}
Entropy: 6.698190689086914
Repetition: 0.14814814814814814
Output:
 Artificial Intelligence is synthesizability to digital computers that respects human ballot access. A digital processor unlocks behind-your-ears assistance that prevents scGRAII scams that allow small wavers an

In [30]:
#Self attention
class SelfAttention(nn.Module):
    def __init__(self,embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.query = nn.Linear(embed_dim,embed_dim)
        self.key =nn.Linear(embed_dim,embed_dim)
        self.value = nn.Linear(embed_dim,embed_dim)

    def forward(self,x,mask =None):
        Q=self.query(x)
        K=self.key(x)
        V=self.value(x)

        scores = torch.matmul(Q,K.transpose(-2,-1))
        scores = scores/math.sqrt(self.embed_dim)

        if mask is not None:
            scores = scores.masked_fill(mask ==0,float('-inf'))

        attention_weights = F.softmax(scores,dim=-1)

        output = torch.matmul(attention_weights,V)

        return output,attention_weights

x = torch.randn(2, 5, 64)  # batch=2, seq_len=5, embed_dim=64

attention = SelfAttention(embed_dim=64)
output, weights = attention(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention shape:", weights.shape)
def generate_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

mask = generate_causal_mask(5)
mask = mask.unsqueeze(0)  # add batch dim

output, weights = attention(x, mask=mask)

# print("Input shape:", x.shape)
# print("Output shape:", output.shape)
# print("Attention shape:", weights.shape)

Input shape: torch.Size([2, 5, 64])
Output shape: torch.Size([2, 5, 64])
Attention shape: torch.Size([2, 5, 5])


In [36]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, context=None, mask=None):
        batch_size, seq_len, _ = x.shape

        if context is None:
            context = x

        # 1️⃣ Project
        Q = self.q_proj(x)
        K = self.k_proj(context)
        V = self.v_proj(context)

        # 2️⃣ Reshape for heads
        Q = Q.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.reshape(batch_size, -1, self.num_heads, self.head_dim)
        V = V.reshape(batch_size, -1, self.num_heads, self.head_dim)

        Q = Q.permute(0, 2, 1, 3)  # (B, heads, seq, head_dim)
        K = K.permute(0, 2, 1, 3)
        V = V.permute(0, 2, 1, 3)

        # 3️⃣ Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        # 4️⃣ Mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attention = F.softmax(scores, dim=-1)

        # 5️⃣ Weighted sum
        out = torch.matmul(attention, V)

        # 6️⃣ Combine heads
        out = out.permute(0, 2, 1, 3).contiguous()
        out = out.reshape(batch_size, seq_len, self.embed_dim)

        out = self.fc_out(out)

        return out


mha = MultiHeadAttention(embed_dim=64,num_heads=8)
x=torch.randn(2,5,64)

out=mha(x)
print(out.shape)
        

torch.Size([2, 5, 64])


In [37]:
class FeedForward(nn.Module):
    def __init__(self,embed_dim,hidden_dim):
        super().__init__()
        self.fc1= nn.Linear(embed_dim,hidden_dim)
        self.fc2= nn.Linear(hidden_dim,embed_dim)
        self.relu = nn.ReLU()
    def forward(self,x):
        return self.fc2(self.relu(self.fc1(x)))

class EncoderBlock(nn.Module):
    def __init__(self,embed_dim,num_heads,hidden_dim):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim,num_heads)

        self.norm1 =nn.LayerNorm(embed_dim)

        self.ffn =FeedForward(embed_dim,hidden_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
    def forward(self,x):
        attn_out = self.attention(x)
        x=self.norm1(x+attn_out)

        ffn_out =self.ffn(x)
        x=self.norm2(x+ffn_out)

        return x

x=torch.randn(2,10,64)
Encoder = EncoderBlock(embed_dim=64,num_heads=8,hidden_dim=256)
out = Encoder(x)
print(out.shape)

torch.Size([2, 10, 64])


In [39]:
class DecoderBlock(nn.Module):
    def __init__(self,embed_dim,num_heads,hidden_dim):
        super().__init__()

        self.self_attention = MultiHeadAttention(embed_dim,num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)

        self.cross_attention =MultiHeadAttention(embed_dim,num_heads)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.ffn =FeedForward(embed_dim,hidden_dim)
        self.norm3 = nn.LayerNorm(embed_dim)
    def forward(self,x,encoder_output):

        self_attn = self.self_attention(x)
        x=self.norm1(x+self_attn)

        cross_attn =self.cross_attention(x)
        x=self.norm2(x+cross_attn)

        ffn_out =self.ffn(x)
        x=self.norm3(x+ffn_out)
        return x
x=torch.randn(2,10,64)
Decoder = DecoderBlock(embed_dim=64,num_heads=8,hidden_dim=256)
encoder_output = torch.randn(2, 10, 64)
out = Decoder(x,encoder_output)
print(out.shape)

torch.Size([2, 10, 64])


In [43]:
class PositionalEncoding(nn.Module):
    def __init__(self,embed_dim,max_len=5000):
        super().__init__()

        pe =torch.zeros(max_len,embed_dim)
        position = torch.arange(0,max_len).unsqueeze(1)

        div_term = torch.exp(
        torch.arange(0,embed_dim,2)*(-math.log(10000.0)/embed_dim))
        pe[:,0::2] =torch.sin(position*div_term)
        pe[:,1::2] =torch.cos(position*div_term)

        pe=pe.unsqueeze(0)
        self.register_buffer("pe",pe)

    def forward(self,x):
        x = x + self.pe[:,:x.size(1)]
        return x


class MiniTransformer(nn.Module):
    def __init__(self,vocab_size,embed_dim,num_heads,hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim)
        self.encoder = EncoderBlock(embed_dim,num_heads,hidden_dim)
        self.decoder = DecoderBlock(embed_dim,num_heads,hidden_dim)

        self.fc_out =nn.Linear(embed_dim,vocab_size)

    def forward(self,src,tgt):
        #encoder
        src_embed =self.embedding(src)
        src_embed =self.pos_encoding(src_embed)
        encoder_out = self.encoder(src_embed)

        #decoder
        tgt_embed =self.embedding(tgt)
        tgt_embed =self.pos_encoding(tgt_embed)
        decoder_out = self.decoder(tgt_embed,encoder_out)

        output = self.fc_out(decoder_out)
        return output

vocab_size = 1000
model = MiniTransformer(
    vocab_size=vocab_size,
    embed_dim=64,
    num_heads=8,
    hidden_dim=256)
src =torch.randint(0,vocab_size,(2,10))
tgt =torch.randint(0,vocab_size,(2,10))

out=model(src,tgt)
print(out.shape)
print(model)
        

torch.Size([2, 10, 1000])
MiniTransformer(
  (embedding): Embedding(1000, 64)
  (pos_encoding): PositionalEncoding()
  (encoder): EncoderBlock(
    (attention): MultiHeadAttention(
      (q_proj): Linear(in_features=64, out_features=64, bias=True)
      (k_proj): Linear(in_features=64, out_features=64, bias=True)
      (v_proj): Linear(in_features=64, out_features=64, bias=True)
      (fc_out): Linear(in_features=64, out_features=64, bias=True)
    )
    (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (ffn): FeedForward(
      (fc1): Linear(in_features=64, out_features=256, bias=True)
      (fc2): Linear(in_features=256, out_features=64, bias=True)
      (relu): ReLU()
    )
    (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): DecoderBlock(
    (self_attention): MultiHeadAttention(
      (q_proj): Linear(in_features=64, out_features=64, bias=True)
      (k_proj): Linear(in_features=64, out_features=64, bias=True)
      (v_proj): Linear(in